In [1]:
# =====================================================================
# Phase 5.5 — Feature Extraction FINAL (BioRob)
# ---------------------------------------------------------------------
# Input:
#   /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/exports_v1_balanced_foldK/
#
# Output:
#   /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/
#       features_v1_eeg_psd_full_foldK_train.npz
#       features_v1_eeg_psd_full_foldK_val.npz
#       features_v1_eeg_psd_full_foldK_test.npz
#
# Important:
#   - This code reads only Phase 5 exported NPZ shards.
#   - It does not read or modify any CSV files.
#   - It does not create new input columns.
#   - The 104 missing Phase 1C files are already excluded by Phase 3/4/5.
#   - It validates shard keys and shapes before extracting features.
# =====================================================================

from __future__ import annotations

import json
import math
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import pandas as pd

from scipy.signal import welch

warnings.filterwarnings("ignore")

In [2]:
# =====================================================================
# CELL 1 — Configuration
# =====================================================================

ROOT_DIR = Path("/home/tsultan1/BioRob/Human Subject Data")
DATASET_DIR = ROOT_DIR / "_dataset_icml_v1"

AUDIT_DIR = ROOT_DIR / "_audit_phase5_5_features"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Phase 5 balanced supervised exports
EXPORT_PREFIX_BALANCED = "exports_v1_balanced"

# Feature file prefix used later by Phase 6
FEATURE_PREFIX = "features_v1_eeg_psd_full"

# Expected splits from Phase 5
SPLITS = ["train", "val", "test"]

# Sampling/window settings from Phase 1C/Phase 4/Phase 5
FS = 250.0
WIN_LEN = 500

# Welch PSD settings
WELCH_NPERSEG = 256

# EEG frequency bands
EEG_BANDS: Dict[str, Tuple[float, float]] = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "mu":    (8.0, 13.0),
    "beta":  (13.0, 30.0),
}

EPS = 1e-8

# Run controls
RUN_INPUT_AUDIT = True

# First test one fold. After Fold 1 works, set RUN_ALL_FOLDS=True.
RUN_ONE_FOLD_FIRST = False
RUN_ALL_FOLDS = True
ONE_FOLD_ID = 1

# False = skip feature files that already exist.
# True  = recompute and overwrite existing feature files.
OVERWRITE_FEATURES = True

# Because Phase 5 was rerun with sliding-only windows, old Phase 5.5 feature
# files must be removed/backed up before extracting new features.
# This prevents mixing old sliding+onset-anchor feature files with new sliding-only exports.
CLEAN_PHASE55_FEATURES_BEFORE_RUN = True
BACKUP_OLD_PHASE55_FEATURES = True

# Save metadata arrays from shards into feature NPZ.
# These are not input CSV columns; they are output metadata arrays for alignment checks.
SAVE_METADATA = True

print("ROOT_DIR:", ROOT_DIR)
print("DATASET_DIR:", DATASET_DIR)
print("EXPORT_PREFIX_BALANCED:", EXPORT_PREFIX_BALANCED)
print("FEATURE_PREFIX:", FEATURE_PREFIX)
print("RUN_INPUT_AUDIT:", RUN_INPUT_AUDIT)
print("RUN_ONE_FOLD_FIRST:", RUN_ONE_FOLD_FIRST)
print("RUN_ALL_FOLDS:", RUN_ALL_FOLDS)
print("ONE_FOLD_ID:", ONE_FOLD_ID)
print("OVERWRITE_FEATURES:", OVERWRITE_FEATURES)
print("CLEAN_PHASE55_FEATURES_BEFORE_RUN:", CLEAN_PHASE55_FEATURES_BEFORE_RUN)
print("BACKUP_OLD_PHASE55_FEATURES:", BACKUP_OLD_PHASE55_FEATURES)
print("SAVE_METADATA:", SAVE_METADATA)

ROOT_DIR: /home/tsultan1/BioRob/Human Subject Data
DATASET_DIR: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1
EXPORT_PREFIX_BALANCED: exports_v1_balanced
FEATURE_PREFIX: features_v1_eeg_psd_full
RUN_INPUT_AUDIT: True
RUN_ONE_FOLD_FIRST: False
RUN_ALL_FOLDS: True
ONE_FOLD_ID: 1
OVERWRITE_FEATURES: True
CLEAN_PHASE55_FEATURES_BEFORE_RUN: True
BACKUP_OLD_PHASE55_FEATURES: True
SAVE_METADATA: True


In [3]:
# =====================================================================
# CELL 2 — Clean/backup old Phase 5.5 feature files before overwrite
# =====================================================================
# This cell is intentionally file-level only. It does NOT read or modify CSVs.
# It removes old Phase 5.5 outputs from the active dataset folder so the
# new sliding-only Phase 5 exports cannot mix with old feature files.

from datetime import datetime
import shutil

def clean_old_phase55_feature_files() -> Path | None:
    """
    Backup/remove old feature files before recomputing features.
    Returns the backup directory if files were backed up; otherwise None.
    """
    feature_files = sorted(DATASET_DIR.glob(f"{FEATURE_PREFIX}_fold*_*.npz"))

    print("=" * 100)
    print("PHASE 5.5 OLD FEATURE CLEANUP")
    print("=" * 100)
    print("Feature pattern:", f"{FEATURE_PREFIX}_fold*_*.npz")
    print("Old feature files found:", len(feature_files))

    for f in feature_files[:10]:
        print(" -", f)
    if len(feature_files) > 10:
        print(" ...")

    if not CLEAN_PHASE55_FEATURES_BEFORE_RUN:
        print("⚠️ CLEAN_PHASE55_FEATURES_BEFORE_RUN=False. Old feature files were kept.")
        return None

    if len(feature_files) == 0:
        print("✅ No old feature files found. Nothing to clean.")
        return None

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_dir = DATASET_DIR / f"_backup_phase55_features_{timestamp}"

    if BACKUP_OLD_PHASE55_FEATURES:
        backup_dir.mkdir(parents=True, exist_ok=True)

    for f in feature_files:
        if BACKUP_OLD_PHASE55_FEATURES:
            target = backup_dir / f.name
            print("Backing up:", f.name, "->", target)
            shutil.move(str(f), str(target))
        else:
            print("Deleting:", f)
            f.unlink()

    print("\n✅ Old Phase 5.5 feature files removed from active dataset folder.")
    if BACKUP_OLD_PHASE55_FEATURES:
        print("Backup folder:", backup_dir)
        return backup_dir

    return None

# Run cleanup before feature extraction if full overwrite is requested.
phase55_backup_dir = clean_old_phase55_feature_files()


PHASE 5.5 OLD FEATURE CLEANUP
Feature pattern: features_v1_eeg_psd_full_fold*_*.npz
Old feature files found: 3
 - /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/features_v1_eeg_psd_full_fold1_test.npz
 - /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/features_v1_eeg_psd_full_fold1_train.npz
 - /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/features_v1_eeg_psd_full_fold1_val.npz
Backing up: features_v1_eeg_psd_full_fold1_test.npz -> /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/_backup_phase55_features_20260526_085654/features_v1_eeg_psd_full_fold1_test.npz
Backing up: features_v1_eeg_psd_full_fold1_train.npz -> /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/_backup_phase55_features_20260526_085654/features_v1_eeg_psd_full_fold1_train.npz
Backing up: features_v1_eeg_psd_full_fold1_val.npz -> /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/_backup_phase55_features_20260526_085654/features_v1_eeg_psd_full_fold1_val.npz

✅ O

In [4]:
# =====================================================================
# CELL 2 — Utility functions
# =====================================================================

def natural_fold_sort_key(path: Path) -> int:
    name = path.name
    if "fold" not in name:
        return 10**9
    try:
        return int(name.split("fold")[-1])
    except Exception:
        return 10**9


def discover_folds() -> List[int]:
    folds = []
    for p in DATASET_DIR.glob(f"{EXPORT_PREFIX_BALANCED}_fold*"):
        if not p.is_dir():
            continue
        try:
            fid = int(p.name.split("fold")[-1])
            folds.append(fid)
        except Exception:
            pass
    return sorted(set(folds))


def fold_dir(fold_id: int) -> Path:
    return DATASET_DIR / f"{EXPORT_PREFIX_BALANCED}_fold{fold_id}"


def split_dir(fold_id: int, split: str) -> Path:
    return fold_dir(fold_id) / split


def feature_out_path(fold_id: int, split: str) -> Path:
    return DATASET_DIR / f"{FEATURE_PREFIX}_fold{fold_id}_{split}.npz"


def list_shards(fold_id: int, split: str) -> List[Path]:
    sd = split_dir(fold_id, split)
    if not sd.exists():
        return []
    return sorted(sd.glob(f"{split}_shard_*.npz"))


def safe_log(x, eps: float = EPS):
    return np.log(np.maximum(np.asarray(x, dtype=np.float64), eps))


def finite_or_zero(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)


def require_keys(z: np.lib.npyio.NpzFile, required: List[str], path: Path):
    missing = [k for k in required if k not in z.files]
    if missing:
        raise KeyError(f"{path} missing required keys: {missing}. Available keys: {z.files}")

In [5]:
# =====================================================================
# CELL 3 — Audit Phase 5 exported shards
# =====================================================================

REQUIRED_SHARD_KEYS = [
    "X_EEG", "X_EMG", "X_ET",
    "y_action", "y_task",
    "subject_id", "task_code", "trial_id", "fold_id",
    "start_idx", "end_idx", "win_len", "source_file",
]

OPTIONAL_SHARD_KEYS = ["win_type"]


def audit_phase5_exports() -> pd.DataFrame:
    """
    Audits Phase 5 exported NPZ shards.
    This checks arrays only. It does not read or modify CSV columns.
    """
    rows = []
    folds = discover_folds()

    print("=" * 100)
    print("PHASE 5.5 INPUT AUDIT — Phase 5 shard folders")
    print("=" * 100)
    print("Folds found:", folds)

    for fid in folds:
        for split in SPLITS:
            shards = list_shards(fid, split)

            if len(shards) == 0:
                rows.append({
                    "fold_id": fid,
                    "split": split,
                    "shard": "",
                    "status": "NO_SHARDS",
                    "n": 0,
                    "eeg_shape": "",
                    "emg_shape": "",
                    "et_shape": "",
                    "error": "",
                })
                continue

            for sp in shards:
                try:
                    z = np.load(sp, allow_pickle=True)
                    require_keys(z, REQUIRED_SHARD_KEYS, sp)

                    X_EEG = z["X_EEG"]
                    X_EMG = z["X_EMG"]
                    X_ET  = z["X_ET"]

                    n = int(X_EEG.shape[0])

                    status = "OK"
                    errors = []

                    if X_EEG.ndim != 3:
                        errors.append(f"X_EEG ndim={X_EEG.ndim}, expected 3")
                    if X_EMG.ndim != 3:
                        errors.append(f"X_EMG ndim={X_EMG.ndim}, expected 3")
                    if X_ET.ndim != 3:
                        errors.append(f"X_ET ndim={X_ET.ndim}, expected 3")

                    if X_EEG.shape[0] != X_EMG.shape[0] or X_EEG.shape[0] != X_ET.shape[0]:
                        errors.append("X_EEG/X_EMG/X_ET window counts do not match")

                    for key in ["y_action", "y_task", "subject_id", "task_code", "trial_id", "fold_id"]:
                        if len(z[key]) != n:
                            errors.append(f"{key} length={len(z[key])}, expected {n}")

                    if X_EEG.ndim == 3 and X_EEG.shape[1] != WIN_LEN:
                        errors.append(f"X_EEG T={X_EEG.shape[1]}, expected {WIN_LEN}")
                    if X_EMG.ndim == 3 and X_EMG.shape[1] != WIN_LEN:
                        errors.append(f"X_EMG T={X_EMG.shape[1]}, expected {WIN_LEN}")

                    if errors:
                        status = "BAD_SHAPE_OR_LENGTH"

                    rows.append({
                        "fold_id": fid,
                        "split": split,
                        "shard": str(sp),
                        "status": status,
                        "n": n,
                        "eeg_shape": str(tuple(X_EEG.shape)),
                        "emg_shape": str(tuple(X_EMG.shape)),
                        "et_shape": str(tuple(X_ET.shape)),
                        "error": "; ".join(errors),
                    })

                except Exception as e:
                    rows.append({
                        "fold_id": fid,
                        "split": split,
                        "shard": str(sp),
                        "status": "ERROR",
                        "n": 0,
                        "eeg_shape": "",
                        "emg_shape": "",
                        "et_shape": "",
                        "error": str(e),
                    })

    audit_df = pd.DataFrame(rows)

    out_path = AUDIT_DIR / "phase5_5_input_shard_audit.csv"
    audit_df.to_csv(out_path, index=False)

    print("\nAudit status counts:")
    print(audit_df["status"].value_counts(dropna=False))
    print("\nSaved:", out_path)

    if (audit_df["status"] != "OK").any():
        print("\n⚠️ Non-OK rows found. First 20:")
        display(audit_df[audit_df["status"] != "OK"].head(20))
    else:
        print("\n✅ All Phase 5 shards passed Phase 5.5 input audit.")

    return audit_df


if RUN_INPUT_AUDIT:
    phase5_5_input_audit = audit_phase5_exports()

PHASE 5.5 INPUT AUDIT — Phase 5 shard folders
Folds found: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]

Audit status counts:
status
OK    57
Name: count, dtype: int64

Saved: /home/tsultan1/BioRob/Human Subject Data/_audit_phase5_5_features/phase5_5_input_shard_audit.csv

✅ All Phase 5 shards passed Phase 5.5 input audit.


In [6]:
# =====================================================================
# CELL 4 — EEG feature functions
# =====================================================================

def compute_eeg_features_window(x: np.ndarray, fs: float = FS) -> np.ndarray:
    """
    Compute deterministic EEG features for one window.

    Input:
        x: (T, C_eeg), already exported by Phase 5.
           This function does not assume original CSV columns.

    Output per channel:
        4 log absolute bandpowers: delta/theta/mu/beta
        4 relative bandpowers: delta/theta/mu/beta
        1 normalized spectral entropy
        3 log Hjorth parameters: activity/mobility/complexity

    Total feature dimension = 12 * C_eeg.
    """
    x = finite_or_zero(x)

    if x.ndim != 2:
        raise ValueError(f"EEG window must be 2D (T,C), got shape {x.shape}")

    T, C = x.shape
    if T < 4 or C < 1:
        return np.zeros((12 * max(C, 1),), dtype=np.float32)

    nperseg = min(WELCH_NPERSEG, T)
    f, Pxx = welch(
        x,
        fs=fs,
        axis=0,
        nperseg=nperseg,
        noverlap=None,
        detrend="constant",
    )

    Pxx = np.nan_to_num(Pxx, nan=0.0, posinf=0.0, neginf=0.0)

    mask_1_30 = (f >= 1.0) & (f <= 30.0)
    if not np.any(mask_1_30):
        # Extremely unlikely for 250 Hz / 500-sample windows, but safe.
        return np.zeros((12 * C,), dtype=np.float32)

    P_1_30 = Pxx[mask_1_30]  # (F, C)
    total_power = P_1_30.sum(axis=0) + EPS

    abs_band_feats = []
    rel_band_feats = []

    for band_name, (lo, hi) in EEG_BANDS.items():
        mask = (f >= lo) & (f <= hi)
        if np.any(mask):
            bp = Pxx[mask].sum(axis=0)
        else:
            bp = np.zeros((C,), dtype=np.float64)

        abs_band_feats.append(safe_log(bp))
        rel_band_feats.append(bp / total_power)

    abs_band = np.stack(abs_band_feats, axis=0)  # (4,C)
    rel_band = np.stack(rel_band_feats, axis=0)  # (4,C)

    P_norm = P_1_30 / (P_1_30.sum(axis=0, keepdims=True) + EPS)
    entropy = -(P_norm * safe_log(P_norm)).sum(axis=0)

    # Normalize entropy to [0,1]-ish by log(number of bins)
    entropy_denom = math.log(max(P_norm.shape[0], 2))
    entropy = entropy / max(entropy_denom, EPS)

    # Hjorth parameters
    dx = np.diff(x, axis=0)
    ddx = np.diff(dx, axis=0)

    var_x = np.var(x, axis=0) + EPS
    var_dx = np.var(dx, axis=0) + EPS
    var_ddx = np.var(ddx, axis=0) + EPS

    activity = var_x
    mobility = np.sqrt(var_dx / var_x)
    mobility_dx = np.sqrt(var_ddx / var_dx)
    complexity = mobility_dx / (mobility + EPS)

    hjorth = np.stack([activity, mobility, complexity], axis=0)
    hjorth = safe_log(hjorth)

    all_feats = np.concatenate(
        [
            abs_band,
            rel_band,
            entropy[None, :],
            hjorth,
        ],
        axis=0,
    )  # (12,C)

    return all_feats.reshape(-1).astype(np.float32)

In [7]:
# =====================================================================
# CELL 5 — EMG feature functions
# =====================================================================

def zero_crossings(sig: np.ndarray, thr: float = 0.01) -> int:
    sig = finite_or_zero(sig).reshape(-1)
    if sig.size < 2:
        return 0
    a = sig[:-1]
    b = sig[1:]
    crosses = (a * b) < 0
    if thr > 0:
        crosses = crosses & (np.abs(a - b) >= thr)
    return int(crosses.sum())


def slope_sign_changes(sig: np.ndarray, thr: float = 0.01) -> int:
    sig = finite_or_zero(sig).reshape(-1)
    if sig.size < 3:
        return 0
    x1 = sig[:-2]
    x2 = sig[1:-1]
    x3 = sig[2:]
    cond = ((x2 - x1) * (x2 - x3)) > 0
    if thr > 0:
        cond = cond & ((np.abs(x2 - x1) >= thr) | (np.abs(x3 - x2) >= thr))
    return int(cond.sum())


def compute_emg_features_window(x: np.ndarray) -> np.ndarray:
    """
    Compute deterministic EMG time-domain features for one window.

    Input:
        x: (T, C_emg), already exported by Phase 5.

    Output per EMG channel:
        RMS, MAV, log waveform length, log zero-crossings,
        log slope-sign changes, log variance.

    Total feature dimension = 6 * C_emg.
    """
    x = finite_or_zero(x)

    if x.ndim != 2:
        raise ValueError(f"EMG window must be 2D (T,C), got shape {x.shape}")

    T, C = x.shape
    if T < 2 or C < 1:
        return np.zeros((6 * max(C, 1),), dtype=np.float32)

    feats = []

    for c in range(C):
        sig = x[:, c]

        rms = np.sqrt(np.mean(sig ** 2))
        mav = np.mean(np.abs(sig))
        wl = np.sum(np.abs(np.diff(sig)))
        zc = zero_crossings(sig, thr=0.01)
        ssc = slope_sign_changes(sig, thr=0.01)
        var_val = np.var(sig)

        feats.extend([
            float(rms),
            float(mav),
            float(np.log1p(max(wl, 0.0))),
            float(np.log1p(max(zc, 0.0))),
            float(np.log1p(max(ssc, 0.0))),
            float(np.log1p(max(var_val, 0.0))),
        ])

    return np.asarray(feats, dtype=np.float32)

In [8]:
# =====================================================================
# CELL 6 — Extract features for one split
# =====================================================================

def collect_metadata_from_shard(z: np.lib.npyio.NpzFile) -> Dict[str, np.ndarray]:
    """
    Collect original metadata arrays from Phase 5 shards.
    These arrays are copied for alignment checks and do not modify input data.
    """
    meta = {}
    if not SAVE_METADATA:
        return meta

    for key in [
        "y_action", "y_task",
        "subject_id", "task_code", "trial_id", "fold_id",
        "win_type", "start_idx", "end_idx", "win_len", "source_file",
    ]:
        if key in z.files:
            meta[key] = z[key]

    return meta


def extract_features_for_split(fold_id: int, split: str) -> Optional[Dict[str, np.ndarray]]:
    """
    Extract EEG PSD/Hjorth features and EMG TD features for one fold/split.

    Returns:
        dict with X_psd, X_emg and metadata arrays.
    """
    shards = list_shards(fold_id, split)

    if len(shards) == 0:
        print(f"[Fold {fold_id}][{split}] No shards found. Skipping.")
        return None

    print("=" * 100)
    print(f"EXTRACTING FEATURES | Fold {fold_id} | Split {split}")
    print("=" * 100)
    print("Shards:", len(shards))

    eeg_feats = []
    emg_feats = []
    meta_buf = defaultdict(list)

    expected_eeg_dim = None
    expected_emg_dim = None
    total_windows = 0

    for shard_idx, sp in enumerate(shards, start=1):
        print(f"  [{shard_idx}/{len(shards)}] {sp.name}")

        z = np.load(sp, allow_pickle=True)
        require_keys(z, REQUIRED_SHARD_KEYS, sp)

        X_EEG = finite_or_zero(z["X_EEG"])
        X_EMG = finite_or_zero(z["X_EMG"])

        if X_EEG.ndim != 3:
            raise ValueError(f"{sp} X_EEG shape must be 3D, got {X_EEG.shape}")
        if X_EMG.ndim != 3:
            raise ValueError(f"{sp} X_EMG shape must be 3D, got {X_EMG.shape}")

        if X_EEG.shape[0] != X_EMG.shape[0]:
            raise ValueError(f"{sp} X_EEG and X_EMG window counts differ: {X_EEG.shape[0]} vs {X_EMG.shape[0]}")

        N, T_eeg, C_eeg = X_EEG.shape
        _, T_emg, C_emg = X_EMG.shape

        if T_eeg != WIN_LEN:
            print(f"    ⚠️ EEG window length {T_eeg} != expected {WIN_LEN}")
        if T_emg != WIN_LEN:
            print(f"    ⚠️ EMG window length {T_emg} != expected {WIN_LEN}")

        if C_eeg < 1:
            raise ValueError(f"{sp} X_EEG has zero channels.")
        if C_emg < 1:
            raise ValueError(f"{sp} X_EMG has zero channels.")

        for i in range(N):
            eeg_feat = compute_eeg_features_window(X_EEG[i], fs=FS)
            emg_feat = compute_emg_features_window(X_EMG[i])

            if expected_eeg_dim is None:
                expected_eeg_dim = eeg_feat.shape[0]
            if expected_emg_dim is None:
                expected_emg_dim = emg_feat.shape[0]

            if eeg_feat.shape[0] != expected_eeg_dim:
                raise ValueError(f"EEG feature dim changed from {expected_eeg_dim} to {eeg_feat.shape[0]} in {sp.name}")
            if emg_feat.shape[0] != expected_emg_dim:
                raise ValueError(f"EMG feature dim changed from {expected_emg_dim} to {emg_feat.shape[0]} in {sp.name}")

            eeg_feats.append(eeg_feat)
            emg_feats.append(emg_feat)

        shard_meta = collect_metadata_from_shard(z)
        for key, arr in shard_meta.items():
            meta_buf[key].append(arr)

        total_windows += N

    if total_windows == 0:
        raise RuntimeError(f"Fold {fold_id} split {split} has zero windows.")

    X_psd = np.stack(eeg_feats, axis=0).astype(np.float32)
    X_emg = np.stack(emg_feats, axis=0).astype(np.float32)

    out = {
        "X_psd": X_psd,
        "X_emg": X_emg,
    }

    for key, chunks in meta_buf.items():
        if len(chunks) > 0:
            out[key] = np.concatenate(chunks, axis=0)

    # Final alignment checks
    n = X_psd.shape[0]
    if X_emg.shape[0] != n:
        raise ValueError("X_psd and X_emg row counts do not match.")

    for key, arr in out.items():
        if key.startswith("X_"):
            continue
        if len(arr) != n:
            raise ValueError(f"Metadata array {key} length={len(arr)} but features N={n}")

    print(f"[Fold {fold_id}][{split}] X_psd shape: {X_psd.shape}")
    print(f"[Fold {fold_id}][{split}] X_emg shape: {X_emg.shape}")

    return out

In [9]:
# =====================================================================
# CELL 7 — Save one split and verify output
# =====================================================================

def save_features_for_split(fold_id: int, split: str) -> Dict:
    out_path = feature_out_path(fold_id, split)

    if out_path.exists() and not OVERWRITE_FEATURES:
        print(f"[Fold {fold_id}][{split}] Existing feature file found, skipped: {out_path}")
        z = np.load(out_path, allow_pickle=True)
        return {
            "fold_id": fold_id,
            "split": split,
            "status": "SKIP_EXISTING",
            "out_path": str(out_path),
            "n": int(z["X_psd"].shape[0]) if "X_psd" in z.files else -1,
            "X_psd_shape": str(tuple(z["X_psd"].shape)) if "X_psd" in z.files else "",
            "X_emg_shape": str(tuple(z["X_emg"].shape)) if "X_emg" in z.files else "",
            "error": "",
        }

    try:
        features = extract_features_for_split(fold_id, split)

        if features is None:
            return {
                "fold_id": fold_id,
                "split": split,
                "status": "NO_SHARDS",
                "out_path": str(out_path),
                "n": 0,
                "X_psd_shape": "",
                "X_emg_shape": "",
                "error": "No shards found.",
            }

        np.savez_compressed(out_path, **features)

        # Reopen immediately for verification
        z = np.load(out_path, allow_pickle=True)

        if "X_psd" not in z.files:
            raise KeyError("Saved feature file missing X_psd")
        if "X_emg" not in z.files:
            raise KeyError("Saved feature file missing X_emg")
        if z["X_psd"].shape[0] != z["X_emg"].shape[0]:
            raise ValueError("Saved X_psd and X_emg row counts do not match.")

        print(f"✅ Saved: {out_path}")

        return {
            "fold_id": fold_id,
            "split": split,
            "status": "OK",
            "out_path": str(out_path),
            "n": int(z["X_psd"].shape[0]),
            "X_psd_shape": str(tuple(z["X_psd"].shape)),
            "X_emg_shape": str(tuple(z["X_emg"].shape)),
            "error": "",
        }

    except Exception as e:
        print(f"❌ ERROR | Fold {fold_id} | Split {split}: {e}")
        return {
            "fold_id": fold_id,
            "split": split,
            "status": "ERROR",
            "out_path": str(out_path),
            "n": 0,
            "X_psd_shape": "",
            "X_emg_shape": "",
            "error": str(e),
        }

In [10]:
# =====================================================================
# CELL 8 — Run Phase 5.5
# =====================================================================

def run_phase5_5(folds: List[int]) -> pd.DataFrame:
    rows = []

    print("=" * 100)
    print("RUNNING PHASE 5.5 FEATURE EXTRACTION")
    print("=" * 100)
    print("Folds:", folds)
    print("Splits:", SPLITS)

    for fid in folds:
        print("\n" + "#" * 100)
        print(f"FOLD {fid}")
        print("#" * 100)

        for split in SPLITS:
            result = save_features_for_split(fid, split)
            rows.append(result)

    summary = pd.DataFrame(rows)
    out_csv = AUDIT_DIR / "phase5_5_feature_extraction_summary.csv"
    summary.to_csv(out_csv, index=False)

    print("\n" + "=" * 100)
    print("PHASE 5.5 SUMMARY")
    print("=" * 100)
    print(summary["status"].value_counts(dropna=False))
    print("Saved summary:", out_csv)

    return summary


folds_found = discover_folds()
print("Folds found:", folds_found)

if len(folds_found) == 0:
    raise FileNotFoundError(
        f"No Phase 5 balanced export folders found in {DATASET_DIR} "
        f"with pattern {EXPORT_PREFIX_BALANCED}_fold*"
    )

if RUN_ONE_FOLD_FIRST:
    if ONE_FOLD_ID not in folds_found:
        raise ValueError(f"ONE_FOLD_ID={ONE_FOLD_ID} not found. Available folds: {folds_found}")
    phase5_5_summary = run_phase5_5([ONE_FOLD_ID])

elif RUN_ALL_FOLDS:
    phase5_5_summary = run_phase5_5(folds_found)

else:
    print("No run selected. Set RUN_ONE_FOLD_FIRST=True or RUN_ALL_FOLDS=True.")

Folds found: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
RUNNING PHASE 5.5 FEATURE EXTRACTION
Folds: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Splits: ['train', 'val', 'test']

####################################################################################################
FOLD 1
####################################################################################################
EXTRACTING FEATURES | Fold 1 | Split train
Shards: 1
  [1/1] train_shard_0001.npz
[Fold 1][train] X_psd shape: (900, 96)
[Fold 1][train] X_emg shape: (900, 24)
✅ Saved: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/features_v1_eeg_psd_full_fold1_train.npz
EXTRACTING FEATURES | Fold 1 | Split val
Shards: 1
  [1/1] val_shard_0001.npz
[Fold 1][val] X_psd shape: (2507, 96)
[Fold 1][val] X_emg shape: (2507, 24)
✅ Saved: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/features_v1_eeg_psd_full_fold1_val.npz
EXTRACTING FEATURES | Fold 1 | Spl

In [11]:
# =====================================================================
# CELL 9 — Verify generated feature files
# =====================================================================

def verify_feature_outputs() -> pd.DataFrame:
    rows = []
    folds = discover_folds()

    for fid in folds:
        for split in SPLITS:
            out_path = feature_out_path(fid, split)

            if not out_path.exists():
                rows.append({
                    "fold_id": fid,
                    "split": split,
                    "status": "MISSING",
                    "out_path": str(out_path),
                    "n": 0,
                    "X_psd_shape": "",
                    "X_emg_shape": "",
                    "error": "Feature file missing.",
                })
                continue

            try:
                z = np.load(out_path, allow_pickle=True)

                if "X_psd" not in z.files or "X_emg" not in z.files:
                    raise KeyError(f"Missing X_psd or X_emg. Keys={z.files}")

                X_psd = z["X_psd"]
                X_emg = z["X_emg"]

                if X_psd.ndim != 2:
                    raise ValueError(f"X_psd ndim={X_psd.ndim}, expected 2")
                if X_emg.ndim != 2:
                    raise ValueError(f"X_emg ndim={X_emg.ndim}, expected 2")
                if X_psd.shape[0] != X_emg.shape[0]:
                    raise ValueError(f"N mismatch: X_psd={X_psd.shape[0]}, X_emg={X_emg.shape[0]}")

                # If metadata exists, verify same length
                meta_errors = []
                for key in ["y_action", "y_task", "subject_id", "task_code", "trial_id"]:
                    if key in z.files and len(z[key]) != X_psd.shape[0]:
                        meta_errors.append(f"{key} len={len(z[key])}, expected {X_psd.shape[0]}")

                status = "OK" if not meta_errors else "META_LENGTH_ERROR"

                rows.append({
                    "fold_id": fid,
                    "split": split,
                    "status": status,
                    "out_path": str(out_path),
                    "n": int(X_psd.shape[0]),
                    "X_psd_shape": str(tuple(X_psd.shape)),
                    "X_emg_shape": str(tuple(X_emg.shape)),
                    "error": "; ".join(meta_errors),
                })

            except Exception as e:
                rows.append({
                    "fold_id": fid,
                    "split": split,
                    "status": "ERROR",
                    "out_path": str(out_path),
                    "n": 0,
                    "X_psd_shape": "",
                    "X_emg_shape": "",
                    "error": str(e),
                })

    verify_df = pd.DataFrame(rows)
    out_csv = AUDIT_DIR / "phase5_5_feature_output_verification.csv"
    verify_df.to_csv(out_csv, index=False)

    print("=" * 100)
    print("PHASE 5.5 OUTPUT VERIFICATION")
    print("=" * 100)
    print(verify_df["status"].value_counts(dropna=False))
    print("Saved:", out_csv)

    bad = verify_df[verify_df["status"] != "OK"]
    if len(bad) == 0:
        print("\n✅ PHASE 5.5 FEATURE FILES VERIFIED CORRECTLY.")
    else:
        print("\n❌ Some feature outputs are missing or invalid:")
        display(bad.head(20))

    return verify_df


phase5_5_verify = verify_feature_outputs()

PHASE 5.5 OUTPUT VERIFICATION
status
OK    57
Name: count, dtype: int64
Saved: /home/tsultan1/BioRob/Human Subject Data/_audit_phase5_5_features/phase5_5_feature_output_verification.csv

✅ PHASE 5.5 FEATURE FILES VERIFIED CORRECTLY.
